# MSW Dot Asset: ComfyUI Colab Launcher

아래 3개 셀을 순서대로 실행합니다.

1. 설치
2. 모델 다운로드
3. ComfyUI 실행 및 접속 링크 생성

각 셀을 나눠서, 어디에서 막혔는지 바로 보이게 했습니다.

In [ ]:
# [1/3] ComfyUI 설치
from pathlib import Path
import subprocess
import sys

ROOT = Path('/content')
COMFY_DIR = ROOT / 'ComfyUI'
CUSTOM_DIR = COMFY_DIR / 'custom_nodes'

def run(cmd, cwd=None, check=True):
    print(f'\n$ {cmd}', flush=True)
    result = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None)
    if check and result.returncode != 0:
        raise RuntimeError(f'명령 실패: {cmd}')
    return result.returncode

def clone_once(url, target):
    target = Path(target)
    if target.exists():
        print(f'이미 있음: {target}')
        return
    run(f'git clone --depth 1 {url} "{target}"')

print('GPU 확인')
run('nvidia-smi', check=False)

print('\nComfyUI와 커스텀 노드를 설치합니다.')
clone_once('https://github.com/comfyanonymous/ComfyUI', COMFY_DIR)
CUSTOM_DIR.mkdir(parents=True, exist_ok=True)
clone_once('https://github.com/ltdrdata/ComfyUI-Manager', CUSTOM_DIR / 'ComfyUI-Manager')
clone_once('https://github.com/Fannovel16/comfyui_controlnet_aux', CUSTOM_DIR / 'comfyui_controlnet_aux')

print('\nPython 패키지를 설치합니다. 이 단계는 몇 분 걸릴 수 있습니다.')
run(f'{sys.executable} -m pip install -U pip')
run(f'{sys.executable} -m pip install xformers torchvision torchaudio')
run(f'{sys.executable} -m pip install -r requirements.txt', cwd=COMFY_DIR)

print('\n[완료] 설치가 끝났습니다. 다음 셀로 넘어가세요.')


In [ ]:
# [2/3] 모델 다운로드
from pathlib import Path
import subprocess

COMFY_DIR = Path('/content/ComfyUI')

if not COMFY_DIR.exists():
    raise RuntimeError('ComfyUI 폴더가 없습니다. 먼저 [1/3] 설치 셀을 실행하세요.')

def run(cmd, cwd=None, check=True):
    print(f'\n$ {cmd}', flush=True)
    result = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None)
    if check and result.returncode != 0:
        raise RuntimeError(f'명령 실패: {cmd}')
    return result.returncode

def download(url, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists() and output_path.stat().st_size > 1024 * 1024:
        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f'이미 있음: {output_path.name} ({size_mb:.1f} MB)')
        return
    run(f'wget -c --show-progress -O "{output_path}" "{url}"')

print('STEP 5 마스크 단독 인페인팅 테스트용 기본 모델을 다운로드합니다.')
download(
    'https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt',
    COMFY_DIR / 'models/checkpoints/sd-v1-5-inpainting.ckpt'
)

# STEP 5에서는 마스크 단독 테스트가 우선입니다.
# ControlNet/IP-Adapter는 다음 단계에서 필요할 때만 받습니다.
DOWNLOAD_OPTIONAL_MODELS = False

if DOWNLOAD_OPTIONAL_MODELS:
    download(
        'https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.safetensors',
        COMFY_DIR / 'models/controlnet/control_v11p_sd15_canny.safetensors'
    )
    download(
        'https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors',
        COMFY_DIR / 'models/clip_vision/CLIP-ViT-H-14-laion2B-s32b-b79K.safetensors'
    )
    download(
        'https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter_sd15.safetensors',
        COMFY_DIR / 'models/ipadapter/ip-adapter_sd15.safetensors'
    )

print('\n다운로드된 체크포인트 목록:')
for path in sorted((COMFY_DIR / 'models/checkpoints').glob('*')):
    print(f'- {path.name}: {path.stat().st_size / 1024 / 1024:.1f} MB')

print('\n[완료] 모델 준비가 끝났습니다. 다음 셀로 넘어가세요.')


In [ ]:
# [3/3] ComfyUI 실행 및 외부 링크 생성
from pathlib import Path
import os
import queue
import re
import subprocess
import sys
import threading
import time
import urllib.request

COMFY_DIR = Path('/content/ComfyUI')
PORT = 8188
LOG_PATH = Path('/content/comfyui.log')
NGROK_AUTHTOKEN = ''  # 여기에 ngrok 토큰을 넣거나, 이전 셀에서 os.environ['NGROK_AUTHTOKEN']로 설정하세요.

if not COMFY_DIR.exists():
    raise RuntimeError('ComfyUI 폴더가 없습니다. 먼저 [1/3] 설치 셀을 실행하세요.')

def print_tail(path, lines=80):
    path = Path(path)
    if not path.exists():
        print('로그 파일이 아직 없습니다.')
        return
    text = path.read_text(errors='replace').splitlines()[-lines:]
    print('\n'.join(text))

def wait_for_comfy(proc, timeout=180):
    started_at = time.time()
    while time.time() - started_at < timeout:
        if proc.poll() is not None:
            print('\nComfyUI가 실행 중 종료되었습니다. 마지막 로그를 확인하세요.')
            print_tail(LOG_PATH)
            raise RuntimeError('ComfyUI 실행 실패')
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2):
                print('ComfyUI 서버 응답 확인 완료')
                return
        except Exception:
            time.sleep(2)
    print_tail(LOG_PATH)
    raise RuntimeError('ComfyUI 서버가 제한 시간 안에 응답하지 않았습니다.')

def stream_process(proc, out_queue):
    for line in proc.stdout:
        out_queue.put(line)

def ensure_pyngrok():
    try:
        from pyngrok import ngrok
        return ngrok
    except ImportError:
        print('pyngrok를 설치합니다.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
        from pyngrok import ngrok
        return ngrok

def start_ngrok_tunnel():
    ngrok = ensure_pyngrok()
    auth_token = os.environ.get('NGROK_AUTHTOKEN', NGROK_AUTHTOKEN).strip()
    if not auth_token:
        raise RuntimeError('NGROK_AUTHTOKEN이 비어 있습니다. 셀 3 상단 변수나 환경변수에 토큰을 넣어주세요.')
    ngrok.kill()
    ngrok.set_auth_token(auth_token)
    tunnel = ngrok.connect(PORT, bind_tls=True)
    return tunnel.public_url

print('기존 8188 포트 관련 프로세스를 정리합니다.')
subprocess.run("pkill -f 'main.py.*8188'", shell=True, check=False)
subprocess.run("pkill -f 'ngrok.*8188'", shell=True, check=False)
time.sleep(2)

print('ComfyUI 서버를 백그라운드로 시작합니다.')
log_file = LOG_PATH.open('w')
comfy_proc = subprocess.Popen(
    [sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', str(PORT)],
    cwd=str(COMFY_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

wait_for_comfy(comfy_proc)

print('\nngrok 링크를 생성합니다.')
try:
    public_url = start_ngrok_tunnel()
    print('\n접속 링크:')
    print(public_url)
except Exception as exc:
    print(f'\nngrok 링크 생성 실패: {exc}')
    print('ComfyUI 로그 마지막 부분:')
    print_tail(LOG_PATH)
    raise

print('\n[완료] ComfyUI는 백그라운드에서 실행 중입니다.')
